In [0]:


# ── Configuration ─────────────────────────────────────────────────────────────
SOURCE_BASE_PATH = "abfss://raw@hospitaldatalake1.dfs.core.windows.net/Hospital_data/MudunuriLokeshSreeSrinivasVarma/Azure-Data-Engineering-Project/refs/heads/main/Hospital_datasets/"
BRONZE_BASE_PATH = "abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/"

TABLE_CONFIG = {
    "appointments": {"file": "appointments.csv", "header": True},
    "billing":      {"file": "billing.csv",       "header": True},
    "doctors":      {"file": "doctors.csv",       "header": True},
    "patients":     {"file": "patients.csv",      "header": True},
    "treatments":   {"file": "treatments.csv",    "header": True},
}

# ── Process ALL tables from raw → bronze ──────────────────────────────────────
for table_name, config in TABLE_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"[BRONZE] Processing: {table_name}")
    print(f"{'='*60}")

    # Step 1 : Read raw CSV
    raw_df = (
        spark.read
             .option("header", config["header"])
             .option("inferSchema", "true")
             .csv(SOURCE_BASE_PATH + config["file"])
    )

    # Step 2 : Add audit columns
    raw_df = (
        raw_df
        .withColumn("_source_file",    F.lit(config["file"]))
        .withColumn("_ingestion_ts",   F.current_timestamp())
        .withColumn("_batch_date",     F.current_date())
        .withColumn("_pipeline_layer", F.lit("BRONZE"))
    )

    row_count = raw_df.count()
    print(f"[BRONZE] Row count: {row_count}")

    # Step 3 : Write to Delta
    bronze_table_path = f"{BRONZE_BASE_PATH}{table_name}"

    (
        raw_df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .save(bronze_table_path)
    )

    print(f"[BRONZE] Written to: {bronze_table_path}")

print(f"\n{'='*60}")
print("[BRONZE] ALL tables processed successfully!")
print(f"{'='*60}")


[BRONZE] Processing: appointments
[BRONZE] Row count: 200
[BRONZE] ✅ Written to: abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/appointments

[BRONZE] Processing: billing
[BRONZE] Row count: 200
[BRONZE] ✅ Written to: abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/billing

[BRONZE] Processing: doctors
[BRONZE] Row count: 10
[BRONZE] ✅ Written to: abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/doctors

[BRONZE] Processing: patients
[BRONZE] Row count: 50
[BRONZE] ✅ Written to: abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/patients

[BRONZE] Processing: treatments
[BRONZE] Row count: 200
[BRONZE] ✅ Written to: abfss://bronze@hospitaldatalake1.dfs.core.windows.net/Hospital_data/treatments

[BRONZE] ✅ ALL tables processed successfully!
